# 17.3 蒙特卡洛方法 / Monte Carlo Methods

**中文**：动态规划(17.2)需要**完全已知**环境模型($P$ 和 $R$)。但现实中我们通常**不知道**环境规律——只能**去玩、去试**，从经历中学习。**蒙特卡洛(Monte Carlo, MC)方法** 是最直接的**无模型(model-free)** 思路:*"要估计一个状态有多好，就从它出发玩很多局到底，把实际拿到的回报平均一下。"* 大数定律保证:采样够多，平均回报就收敛到真实价值。本节用**21点(Blackjack)** 演示——一个我们不需要写出概率公式、只需"发牌开玩"的环境。
**English**: Dynamic programming (17.2) needs a **fully known** model ($P$ and $R$). But in reality we usually **don't know** the environment's dynamics — we can only **play and try**, learning from experience. **Monte Carlo (MC) methods** are the most direct **model-free** idea: *"to estimate how good a state is, play many complete episodes from it and average the actual returns received."* The law of large numbers guarantees: with enough samples, the average return converges to the true value. We demonstrate on **Blackjack** — an environment where we never write a probability formula, just deal and play.

---

**中文**：MC 的核心:**价值 = 回报的样本平均**。要评估策略 $\pi$ 的 $V^\pi(s)$:
**English**: MC's core: **value = sample average of returns**. To evaluate $V^\pi(s)$ for policy $\pi$:
1. 用 $\pi$ 玩很多**完整回合(episode)**，直到终止。
   Play many **complete episodes** with $\pi$ until termination.
2. 对每个回合，从每个访问过的状态 $s$ 往后算实际回报 $G_t$。
   For each episode, compute the actual return $G_t$ following each visited state $s$.
3. $V^\pi(s)$ = 所有这些 $G_t$ 的平均。
   $V^\pi(s)$ = the average of all those $G_t$.

**中文**：两种统计口径:
**English**: Two counting schemes:
- **首次访问(first-visit)**:一个回合里，一个状态只用它**第一次**出现后的回报。
  **First-visit**: within an episode, use only the return following a state's **first** occurrence.
- **每次访问(every-visit)**:状态每次出现都算一次。
  **Every-visit**: count every occurrence of the state.

**中文**：要找**最优策略**(MC 控制)，把 MC 评估和**改进**结合:估计**动作价值 $Q(s,a)$**(而非 $V$，因为无模型时不知道 $P$，光有 $V$ 无法贪心选动作)，再用 **ε-贪心** 改进——大部分时候选当前最好动作，偶尔($\varepsilon$ 概率)随机探索，保证每个动作都被试到。
**English**: To find the **optimal policy** (MC control), combine MC evaluation with **improvement**: estimate **action values $Q(s,a)$** (not $V$ — model-free, we don't know $P$, so $V$ alone can't pick actions greedily), then improve with **ε-greedy** — mostly take the current best action, occasionally (prob $\varepsilon$) explore randomly, ensuring every action gets tried.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 无模型入门必考）**
> **中文**：MC=**无模型**、从**完整回合**的实际回报采样平均来估计价值。**首次访问 vs 每次访问**(都收敛)。**MC 控制**:估 Q(s,a) + ε-贪心改进(GLIE:ε 渐降到0则收敛到最优)。**vs DP**:MC 不需模型、不用自举(bootstrap), 直接用真实回报; 但**只适用回合制任务**(必须能结束)、**方差大**(整条轨迹的随机性都进了回报)、**每回合结束才更新**(慢)。**关键**:无模型必须估 Q 而非 V(否则无法贪心选动作); 探索靠 ε-贪心或 exploring starts。
> **English**: MC = **model-free**, estimating value by averaging actual returns from **complete episodes**. **First-visit vs every-visit** (both converge). **MC control**: estimate Q(s,a) + ε-greedy improvement (GLIE: ε decaying to 0 → converges to optimal). **vs DP**: MC needs no model and no bootstrapping — it uses real returns; but works **only for episodic tasks** (must terminate), has **high variance** (all trajectory randomness enters the return), and **updates only at episode end** (slow). **Key**: model-free must estimate Q not V (else can't act greedily); exploration via ε-greedy or exploring starts.


In [ ]:

# ============================================================
# 环境:从零实现 21点 Blackjack / Blackjack from scratch (Sutton & Barto)
# 中文:玩家想让手牌点数尽量接近21但不超过。A可当1或11(可用A=usable ace)。
#      状态=(玩家点数12-21, 庄家明牌1-10, 是否有可用A)。动作:0=停牌stick, 1=要牌hit。
#      玩家停牌后庄家按固定规则(<17就要牌)行动。赢+1 输-1 平0。
# English: player aims close to 21 without busting. Ace counts 1 or 11 (usable ace).
#      State=(player sum 12-21, dealer's up card 1-10, usable ace). Actions: 0=stick, 1=hit.
#      After the player sticks, the dealer hits until >=17. Win +1, lose -1, draw 0.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from collections import defaultdict
rng=np.random.default_rng(0)
def draw(): return min(int(rng.integers(1,14)),10)          # 抽一张牌 1..13 -> 花牌算10 / draw a card
def hand_value(cards):                                       # 手牌点数 + 是否用了可用A / value + usable-ace
    s=sum(cards); usable = (1 in cards) and (s+10<=21)
    return (s+10, True) if usable else (s, False)

def play_episode(policy):
    """用 policy 玩一整局, 返回 [(状态,动作),...] 和最终奖励 / play one full hand."""
    player=[draw(),draw()]; dealer=[draw(),draw()]; episode=[]
    while True:                                              # 玩家回合 / player's turn
        ps,us=hand_value(player)
        if ps<12: player.append(draw()); continue           # <12 必然要牌(无需决策)/ auto-hit below 12
        state=(ps, dealer[0], int(us)); a=policy(state); episode.append((state,a))
        if a==1:                                            # 要牌 / hit
            player.append(draw())
            if hand_value(player)[0]>21: return episode,-1.0 # 爆牌=输 / bust -> lose
        else: break                                         # 停牌 / stick
    while hand_value(dealer)[0]<17: dealer.append(draw())    # 庄家回合 / dealer hits until >=17
    pv,dv=hand_value(player)[0], hand_value(dealer)[0]
    return episode, (1.0 if (dv>21 or pv>dv) else (-1.0 if pv<dv else 0.0))

# 快速 sanity check / quick sanity check
random_pol=lambda s: int(rng.integers(2))
ep,rew=play_episode(random_pol); print("一局示例 / a sample hand:", ep, "-> 奖励/reward", rew)


**中文**：先做 **MC 预测(policy evaluation)**:评估一个固定的朴素策略——"点数 ≥20 才停牌，否则一直要牌"。我们玩几十万局，对每个状态用**首次访问**平均实际回报，得到 $V^\pi$，再把它画成经典的价值曲面。
**English**: First **MC prediction (policy evaluation)**: evaluate a fixed naive policy — "stick only at 20+, otherwise keep hitting." We play hundreds of thousands of hands and, using **first-visit** averaging of actual returns, obtain $V^\pi$, then plot the classic value surface.


In [ ]:

# ============================================================
# MC 预测:评估"≥20才停牌"策略 / MC prediction of the "stick at 20+" policy
# ============================================================
def stick20(state): return 0 if state[0]>=20 else 1          # ≥20 停牌, 否则要牌 / stick if sum>=20

def mc_predict(policy, n=200000, gamma=1.0):
    V=defaultdict(float); N=defaultdict(int)
    for _ in range(n):
        episode,rew=play_episode(policy)
        G=rew                                                # 只有终局有奖励, gamma=1 / reward only at end
        seen=set()
        for (s,a) in episode:                                # 首次访问 / first-visit
            if s in seen: continue
            seen.add(s); N[s]+=1; V[s]+=(G-V[s])/N[s]        # 增量平均 / incremental average
    return V

V=mc_predict(stick20, 200000)
print(f"评估完成, 访问到 {len(V)} 个状态 / evaluated {len(V)} states")
print("示例 / examples:")
print("  V(玩家20, 庄家明牌6, 无A) =", round(V[(20,6,0)],3), "(接近1: 20点很可能赢)")
print("  V(玩家13, 庄家明牌10,无A) =", round(V[(13,10,0)],3), "(负: 13点对庄家10很吃亏)")


**中文**：再做 **MC 控制**——不给定策略，而是**自己学出最优策略**。用 ε-贪心边玩边改进:估计 $Q(s,a)$，每局按"90% 选当前最优动作、10% 随机探索"来玩，用首次访问增量更新 $Q$。玩够多局后，贪心取 $Q$ 就是最优策略。看它能否学出教科书级的 21点策略。
**English**: Now **MC control** — no given policy; **learn the optimal policy itself**. Play-and-improve with ε-greedy: estimate $Q(s,a)$, play each hand "90% take the current-best action, 10% explore randomly," and update $Q$ by first-visit incremental averaging. After enough hands, greedy over $Q$ is the optimal policy. Does it recover the textbook Blackjack strategy?


In [ ]:

# ============================================================
# MC 控制:ε-贪心 + 首次访问, 学最优策略 / MC control (epsilon-greedy, first-visit)
# ============================================================
def mc_control(n=500000, eps=0.1, gamma=1.0):
    Q=defaultdict(lambda: np.zeros(2)); N=defaultdict(lambda: np.zeros(2))
    def policy(s):                                           # ε-贪心行为策略 / ε-greedy behavior policy
        if rng.random()<eps: return int(rng.integers(2))     # 探索 / explore
        return int(np.argmax(Q[s]))                          # 利用(当前最优)/ exploit
    for _ in range(n):
        episode,rew=play_episode(policy); G=rew; seen=set()
        for (s,a) in episode:                                # 首次访问 (s,a) / first-visit
            if (s,a) in seen: continue
            seen.add((s,a)); N[s][a]+=1; Q[s][a]+=(G-Q[s][a])/N[s][a]   # 增量平均 / incremental mean
    return Q

Qstar=mc_control(500000, eps=0.1)
print(f"MC 控制完成, 学到 {len(Qstar)} 个状态的Q值 / learned Q over {len(Qstar)} states")
# 打印无可用A时的最优策略(1=要牌,0=停牌)/ optimal policy without usable ace
print("\n最优策略(无可用A), 行=玩家点数, 列=庄家明牌1-10 (H=要牌, S=停牌):")
for ps in range(21,11,-1):
    row="".join("H" if int(np.argmax(Qstar[(ps,d,0)]))==1 else "S" for d in range(1,11))
    print(f"  玩家{ps:2d}: {row}")


**中文**：把 MC 预测的**价值曲面**和 MC 控制的**最优策略**都画出来——这正是 Sutton & Barto 教科书里的经典两图。
**English**: Plot both the MC-predicted **value surface** and the MC-controlled **optimal policy** — the classic pair of figures from Sutton & Barto.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(16,4.7))
# ① 价值曲面(无可用A): 行=玩家12-21, 列=庄家1-10 / value surface (no usable ace)
Vgrid=np.array([[V[(ps,d,0)] for d in range(1,11)] for ps in range(21,11,-1)])
im0=ax[0].imshow(Vgrid,cmap="RdYlGn",vmin=-1,vmax=1,aspect="auto")
ax[0].set_yticks(range(10)); ax[0].set_yticklabels(range(21,11,-1)); ax[0].set_xticks(range(10)); ax[0].set_xticklabels(range(1,11))
ax[0].set_title('"≥20停牌"策略的价值V (无A) / V of stick@20'); ax[0].set_xlabel("庄家明牌 dealer"); ax[0].set_ylabel("玩家点数 player")
plt.colorbar(im0,ax=ax[0],fraction=0.046)
# ② 最优策略(无可用A) / optimal policy (no usable ace)
Pgrid=np.array([[int(np.argmax(Qstar[(ps,d,0)])) for d in range(1,11)] for ps in range(21,11,-1)])
ax[1].imshow(Pgrid,cmap="coolwarm",aspect="auto")
ax[1].set_yticks(range(10)); ax[1].set_yticklabels(range(21,11,-1)); ax[1].set_xticks(range(10)); ax[1].set_xticklabels(range(1,11))
ax[1].set_title("最优策略 (无A): 红=要牌H 蓝=停牌S / optimal π"); ax[1].set_xlabel("庄家明牌 dealer"); ax[1].set_ylabel("玩家点数 player")
# ③ 最优策略(有可用A) / optimal policy (usable ace)
Pgrid2=np.array([[int(np.argmax(Qstar[(ps,d,1)])) for d in range(1,11)] for ps in range(21,11,-1)])
ax[2].imshow(Pgrid2,cmap="coolwarm",aspect="auto")
ax[2].set_yticks(range(10)); ax[2].set_yticklabels(range(21,11,-1)); ax[2].set_xticks(range(10)); ax[2].set_xticklabels(range(1,11))
ax[2].set_title("最优策略 (有可用A): 更激进 / optimal π (usable ace)"); ax[2].set_xlabel("庄家明牌 dealer"); ax[2].set_ylabel("玩家点数 player")
plt.tight_layout(); plt.savefig("/tmp/rl03_viz.png",dpi=80); plt.show()
print("MC 学出的策略 = 教科书级 21点基本策略! / MC recovered the textbook basic strategy!")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **MC 从"纯玩"里学出了正确策略**——全程**没有写一行环境的概率公式**，只是发牌、开打、数输赢。学出的最优策略和赌场"基本策略表"高度一致:**17点以上一律停牌**;12-16点时**庄家明牌弱(2-6, 容易爆)就停牌逼庄家爆、庄家明牌强(7-A)就冒险要牌**。有可用A时更激进(A能当1兜底不怕爆)。
2. **价值曲面很直观**:玩家点数越高越接近赢面(绿),点数低+庄家明牌强就吃亏(红)。这是从几十万局的真实胜负里"平均"出来的。
3. **MC 的代价**:①**必须回合结束才能更新**(21点一局很短还好, 但下棋/走迷宫一局很长就慢); ②**方差大**——一局的输赢受运气影响大, 要很多局才平均得准(注意我们用了几十万局); ③**只适用能终止的回合制任务**。这三点正是下一节 **TD 学习**要改进的:每走一步就更新(自举)、方差更小、还能处理连续不终止的任务。

**English**:
1. **MC learned the correct strategy from pure play** — with **not a single environment probability formula**, just dealing, playing, and counting wins/losses. The learned optimal policy closely matches the casino "basic strategy": **always stick at 17+**; on 12-16, **stick when the dealer shows a weak card (2-6, likely to bust) to force a dealer bust, but risk a hit when the dealer shows strong (7-A)**. With a usable ace it plays more aggressively (the ace as 1 cushions against busting).
2. **The value surface is intuitive**: higher player sums are closer to winning (green); low sums + strong dealer card lose (red). Averaged from the real win/loss of hundreds of thousands of hands.
3. **MC's costs**: ① **updates only at episode end** (fine for short Blackjack hands, but slow for long games/mazes); ② **high variance** — one hand's outcome is luck-heavy, needing many hands to average accurately (note we used hundreds of thousands); ③ **works only for terminating, episodic tasks**. These three are exactly what **TD learning** (next) improves: update every step (bootstrapping), lower variance, and handle continuing non-terminating tasks.

> 💼 **实战视角 / Practical angle**
> **中文**:纯 MC 在深度 RL 里较少单独用(方差大、更新慢), 但它的思想无处不在:①**回合级回报**是 REINFORCE(策略梯度, 17.7)的基础; ②**离线评估**、蒙特卡洛树搜索(MCTS, AlphaGo 的核心)都用 MC rollout 估值; ③游戏 AI、A/B 长期价值估计。**首次 vs 每次访问**:每次访问实现更简单、深度 RL 里更常用。面试金句:*"MC 用完整回合的真实回报平均来估值, 无模型、无偏但方差大、只适用回合制; TD 用自举(一步预测)换来低方差和在线更新, 但引入偏差——这就是偏差-方差权衡在 RL 里的体现。"*
> **English**: Pure MC is rarely used alone in deep RL (high variance, slow updates), but its ideas are everywhere: ① **episodic returns** underlie REINFORCE (policy gradients, 17.7); ② **off-policy evaluation** and Monte Carlo Tree Search (MCTS, the heart of AlphaGo) use MC rollouts to estimate value; ③ game AI, long-term A/B value estimation. **First- vs every-visit**: every-visit is simpler to implement and more common in deep RL. Interview line: *"MC averages real returns over complete episodes — model-free and unbiased but high-variance and episodic-only; TD trades a one-step bootstrap for low variance and online updates at the cost of bias — the bias-variance tradeoff manifested in RL."*

---
### 小结 / Summary
- **中文**:MC=无模型, 用完整回合的实际回报采样平均估值; 首次/每次访问都收敛。
- **English**: MC = model-free, estimating value by averaging actual returns over complete episodes; first-/every-visit both converge.
- **中文**:MC 控制=估 Q(s,a)+ε-贪心改进, 在 Blackjack 上无需任何概率公式就学出教科书策略。
- **English**: MC control = estimate Q(s,a) + ε-greedy improvement; on Blackjack it recovers the textbook strategy with no probability formulas.
- **中文**:代价:只适用回合制、方差大、回合末才更新——引出 TD 学习(下节)。
- **English**: Costs: episodic-only, high variance, updates at episode end — motivating TD learning (next).
